# Who is the best driver once you remove the effect of the car?

Method: head-to-head comparison against teammates (same car).
Data: F1DB (CC BY 4.0), up to the 2026 Brazilian Grand Prix.

In [1]:
import duckdb

duckdb.sql("""
    SELECT COUNT(*) AS rows
    FROM '../data/raw/f1db/f1db-races-qualifying-results.csv'
""")

rowsBIGINT
27149


In [2]:
duckdb.sql("""
    SELECT 'qualifying' AS source, COUNT(*) AS rows
    FROM '../data/raw/f1db/f1db-races-qualifying-results.csv'
    UNION ALL
    SELECT 'race', COUNT(*)
    FROM '../data/raw/f1db/f1db-races-race-results.csv'
""")

sourceVARCHAR,rowsBIGINT
qualifying,27149
race,27731


In [3]:
duckdb.sql("""
    SELECT year, COUNT(*) AS rows
    FROM '../data/raw/f1db/f1db-races-qualifying-results.csv'
    GROUP BY year
    ORDER BY year
""")

yearBIGINT,rowsBIGINT
1950,153
1951,174
1952,212
1953,230
1954,200
1955,152
1956,174
1957,160
1958,235
1959,194


In [4]:
duckdb.sql("""
    WITH quali AS (
        SELECT year, COUNT(*) AS quali_rows
        FROM '../data/raw/f1db/f1db-races-qualifying-results.csv'
        GROUP BY year
    ),
    race AS (
        SELECT year, COUNT(*) AS race_rows
        FROM '../data/raw/f1db/f1db-races-race-results.csv'
        GROUP BY year
    )
    SELECT race.year,
           race_rows,
           COALESCE(quali_rows, 0) AS quali_rows,
           race_rows - COALESCE(quali_rows, 0) AS difference
    FROM race
    LEFT JOIN quali ON race.year = quali.year
    WHERE race_rows - COALESCE(quali_rows, 0) <> 0
    ORDER BY race.year
""")

yearBIGINT,race_rowsBIGINT,quali_rowsBIGINT,differenceBIGINT
1950,162,153,9
1951,182,174,8
1952,221,212,9
1953,255,230,25
1954,230,200,30
1955,180,152,28
1956,193,174,19
1957,171,160,11
1958,241,235,6
1959,197,194,3


In [5]:
duckdb.sql("""
    WITH quali AS (
        SELECT year, COUNT(*) AS quali_rows
        FROM '../data/raw/f1db/f1db-races-qualifying-results.csv'
        GROUP BY year
    ),
    race AS (
        SELECT year, COUNT(*) AS race_rows
        FROM '../data/raw/f1db/f1db-races-race-results.csv'
        GROUP BY year
    )
    SELECT (race.year // 10) * 10 AS decade,
           SUM(race_rows - COALESCE(quali_rows, 0)) AS difference
    FROM race
    LEFT JOIN quali ON race.year = quali.year
    GROUP BY decade
    ORDER BY decade
""")

decadeBIGINT,differenceHUGEINT
1950,148
1960,17
1970,34
1980,209
1990,149
2000,4
2010,11
2020,10


In [6]:
duckdb.sql("""
    SELECT (year // 10) * 10 AS decade,
           COUNT(*) AS dnpq_rows
    FROM '../data/raw/f1db/f1db-races-race-results.csv'
    WHERE positionText = 'DNPQ'
    GROUP BY decade
    ORDER BY decade
""")

decadeBIGINT,dnpq_rowsBIGINT
1960,3
1970,30
1980,173
1990,132


In [7]:
duckdb.sql("""
    SELECT (year // 10) * 10 AS decade,
           COUNT(*) AS shared_car_rows
    FROM '../data/raw/f1db/f1db-races-race-results.csv'
    WHERE sharedCar = true
    GROUP BY decade
    ORDER BY decade
""")

decadeBIGINT,shared_car_rowsBIGINT
1950,122
1960,5


In [8]:
duckdb.sql("""
    DESCRIBE SELECT *
    FROM '../data/raw/f1db/f1db-races-qualifying-results.csv'
""")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
raceId,BIGINT,YES,NULL,NULL,NULL
year,BIGINT,YES,NULL,NULL,NULL
round,BIGINT,YES,NULL,NULL,NULL
positionDisplayOrder,BIGINT,YES,NULL,NULL,NULL
positionNumber,BIGINT,YES,NULL,NULL,NULL
positionText,VARCHAR,YES,NULL,NULL,NULL
driverNumber,BIGINT,YES,NULL,NULL,NULL
driverId,VARCHAR,YES,NULL,NULL,NULL
constructorId,VARCHAR,YES,NULL,NULL,NULL
engineManufacturerId,VARCHAR,YES,NULL,NULL,NULL


In [10]:
duckdb.sql("""
SELECT COUNT(*) AS total_rows,
COUNT(positionNumber) AS with_position_number,
COUNT(positionDisplayOrder) AS with_display_Order,
FROM '../data/raw/f1db/f1db-races-qualifying-results.csv'
""")

total_rowsBIGINT,with_position_numberBIGINT,with_display_OrderBIGINT
27149,27012,27149


In [11]:
duckdb.sql("""
SELECT positionText, COUNT(*) AS rows,
FROM '../data/raw/f1db/f1db-races-qualifying-results.csv'
WHERE positionNumber IS NULL
GROUP BY positionText
ORDER BY rows DESC
""")

positionTextVARCHAR,rowsBIGINT
NC,102
EX,23
DSQ,12


## Decisiones de análisis

**Decisión 1: columna de posición.** Usamos `positionNumber`, la posición real en la
clasificación. Descartamos `positionDisplayOrder` porque coloca a los pilotos sin
posición al final sin que se note, lo que esconde una decisión.

**Decisión 2: pilotos sin posición válida.** Hay 137 filas de 27.149 (0,5 %) con
`positionNumber` vacío (NULL): NC 102, EX 23 y DSQ 12. El código no dice si la causa
fue el piloto (un error o un accidente) o el coche (un fallo técnico o una pieza
ilegal).

- **Análisis principal:** eliminamos los duelos entre compañeros en los que al menos
  uno no tiene posición válida, porque no son una comparación en igualdad de
  condiciones.
- **Comprobación (sensibilidad):** repetimos el análisis contando esos duelos como
  derrota del piloto sin posición, porque a veces la culpa sí es suya.
- **Si los dos rankings cambian notablemente**, estudiaremos algunos casos concretos
  para entender la causa de esos NULL.